# Geometría del MCO y Teorema FWL: Prueba de Simulación
## Demostración empírica de la equivalencia entre MCO múltiple y proyecciones
**Simulación basada en el Modelo de Mincer**

In [2]:
# Instalación de librerías necesarias
!pip install plotly kable -q

# Importaciones
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Configuración de estilo
pd.set_option('display.float_format', '{:.6f}'.format)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.7 MB/s eta 0:00:00


# 📌 Paso 1: Simulación del Modelo de Mincer

Simulamos una población de $N=1000$ individuos que sigue la estructura del modelo de Mincer: el logaritmo del salario depende de la educación y la experiencia.

Fijamos una semilla para garantizar que los resultados sean exactamente los mismos cada vez que se ejecute.

In [4]:
# Paso 1: Simulación del Modelo de Mincer
np.random.seed(42)  # Semilla fija para reproducibilidad total
N = 1000

# 1. Variables explicativas (con cierta correlación realista)
educacion = np.random.normal(loc=12, scale=3, size=N)
experiencia = np.random.normal(loc=20, scale=8, size=N) + 0.5 * educacion

# 2. Término de error
u = np.random.normal(loc=0, scale=0.25, size=N)

# 3. Variable dependiente: Modelo de Mincer verdadero
# ln(Salario) = 0.5 + 0.08*Educacion + 0.04*Experiencia + u
ln_salario = 0.5 + 0.08 * educacion + 0.04 * experiencia + u

# Crear DataFrame
df_mincer = pd.DataFrame({
    'ln_salario': ln_salario,
    'educacion': educacion,
    'experiencia': experiencia
})

print("Primeras 6 observaciones de la base simulada:")
df_mincer.head(6)

Primeras 6 observaciones de la base simulada:


,ln_salario,educacion,experiencia
0,2.928013,13.490142,37.939915
1,2.718274,11.585207,33.189673
2,2.515283,13.943066,27.448576
3,2.672899,16.569090,23.109051
4,2.179782,11.297540,31.234556
5,2.608998,11.297589,28.796678


# 📌 Paso 2: La Prueba Algebraica (MCO vs. FWL)

Aquí ponemos a prueba el núcleo del paper. Estimaremos el efecto de la educación ($\beta_1$) de dos formas:

1. **MCO Tradicional**: Regresión múltiple completa.
2. **Teorema FWL (2 pasos)**:
   - Paso A: "Purificar" (residualizar) el salario y la educación, eliminando el efecto de la experiencia.
   - Paso B: Regresar el salario purificado sobre la educación purificada.

In [5]:
# Paso 2: La Prueba Algebraica (MCO vs. FWL)

# ============================================
# CAMINO 1: MCO Tradicional (Regresión Múltiple)
# ============================================
# Usamos álgebra lineal pura: β = (X'X)^-1 X'y
X = np.column_stack([np.ones(N), educacion, experiencia])
Y = ln_salario

betas_mco = np.linalg.inv(X.T @ X) @ X.T @ Y
beta0_mco, beta1_mco, beta2_mco = betas_mco

print("=" * 50)
print("CAMINO 1: MCO Tradicional")
print("=" * 50)
print(f"β₀ (Intercepto): {beta0_mco:.8f}")
print(f"β₁ (Educación):  {beta1_mco:.8f}")
print(f"β₂ (Experiencia): {beta2_mco:.8f}")
print()

# ============================================
# CAMINO 2A: FWL para β1 (Educación)
# ============================================
# Purificar Y y X1 respecto a X2 (Experiencia)
# Regresión auxiliar: Y ~ X2
X2_only = np.column_stack([np.ones(N), experiencia])
beta_y_x2 = np.linalg.inv(X2_only.T @ X2_only) @ X2_only.T @ Y
y_pred_x2 = X2_only @ beta_y_x2
M1_y = Y - y_pred_x2  # Residuos

# Regresión auxiliar: X1 ~ X2
beta_x1_x2 = np.linalg.inv(X2_only.T @ X2_only) @ X2_only.T @ educacion
x1_pred_x2 = X2_only @ beta_x1_x2
M1_x1 = educacion - x1_pred_x2  # Residuos

# Regresión final: M1_y ~ M1_x1 (sin intercepto)
beta1_fwl = (M1_x1 @ M1_y) / (M1_x1 @ M1_x1)

print("=" * 50)
print("CAMINO 2A: FWL para β₁ (Educación)")
print("=" * 50)
print(f"β₁ (FWL): {beta1_fwl:.8f}")
print()

# ============================================
# CAMINO 2B: FWL para β2 (Experiencia)
# ============================================
# Purificar Y y X2 respecto a X1 (Educación)
X1_only = np.column_stack([np.ones(N), educacion])
beta_y_x1 = np.linalg.inv(X1_only.T @ X1_only) @ X1_only.T @ Y
y_pred_x1 = X1_only @ beta_y_x1
M2_y = Y - y_pred_x1  # Residuos

# Regresión auxiliar: X2 ~ X1
beta_x2_x1 = np.linalg.inv(X1_only.T @ X1_only) @ X1_only.T @ experiencia
x2_pred_x1 = X1_only @ beta_x2_x1
M2_x2 = experiencia - x2_pred_x1  # Residuos

# Regresión final: M2_y ~ M2_x2 (sin intercepto)
beta2_fwl = (M2_x2 @ M2_y) / (M2_x2 @ M2_x2)

print("=" * 50)
print("CAMINO 2B: FWL para β₂ (Experiencia)")
print("=" * 50)
print(f"β₂ (FWL): {beta2_fwl:.8f}")
print()

# ============================================
# VERIFICACIÓN DE IGUALDAD
# ============================================
diff_beta1 = abs(beta1_mco - beta1_fwl)
diff_beta2 = abs(beta2_mco - beta2_fwl)

print("=" * 50)
print("VERIFICACIÓN DEL TEOREMA FWL")
print("=" * 50)
print(f"β₁ (Educación):")
print(f"  MCO: {beta1_mco:.8f}")
print(f"  FWL: {beta1_fwl:.8f}")
print(f"  Diferencia: {diff_beta1:.2e}")
print(f"  ¿Iguales? {'✅ SÍ' if diff_beta1 < 1e-10 else ' NO'}")
print()
print(f"β₂ (Experiencia):")
print(f"  MCO: {beta2_mco:.8f}")
print(f"  FWL: {beta2_fwl:.8f}")
print(f"  Diferencia: {diff_beta2:.2e}")
print(f"  ¿Iguales? {'✅ SÍ' if diff_beta2 < 1e-10 else '❌ NO'}")
print()

# ============================================
# TABLA COMPARATIVA FINAL
# ============================================
resultados = pd.DataFrame({
    'Parámetro': ['β₁ (Educación)', 'β₁ (Educación)', 'β₂ (Experiencia)', 'β₂ (Experiencia)'],
    'Método': ['MCO Múltiple', 'FWL (Purificación)', 'MCO Múltiple', 'FWL (Purificación)'],
    'Estimación': [
        f"{beta1_mco:.8f}",
        f"{beta1_fwl:.8f}",
        f"{beta2_mco:.8f}",
        f"{beta2_fwl:.8f}"
    ],
    'Verificación': [
        'Base',
        '✅ IDÉNTICO' if diff_beta1 < 1e-10 else '❌ DIFERENTE',
        'Base',
        '✅ IDÉNTICO' if diff_beta2 < 1e-10 else ' DIFERENTE'
    ]
})

print("Demostración completa del Teorema FWL:")
print(resultados.to_string(index=False))

CAMINO 1: MCO Tradicional
β₀ (Intercepto): 0.48608841
β₁ (Educación):  0.08197617
β₂ (Experiencia): 0.03968196

CAMINO 2A: FWL para β₁ (Educación)
β₁ (FWL): 0.08197617

CAMINO 2B: FWL para β₂ (Experiencia)
β₂ (FWL): 0.03968196

VERIFICACIÓN DEL TEOREMA FWL
β₁ (Educación):
  MCO: 0.08197617
  FWL: 0.08197617
  Diferencia: 8.88e-16
  ¿Iguales? ✅ SÍ

β₂ (Experiencia):
  MCO: 0.03968196
  FWL: 0.03968196
  Diferencia: 2.57e-16
  ¿Iguales? ✅ SÍ

Demostración completa del Teorema FWL:
       Parámetro             Método Estimación Verificación
  β₁ (Educación)       MCO Múltiple 0.08197617         Base
  β₁ (Educación) FWL (Purificación) 0.08197617   ✅ IDÉNTICO
β₂ (Experiencia)       MCO Múltiple 0.03968196         Base
β₂ (Experiencia) FWL (Purificación) 0.03968196   ✅ IDÉNTICO


# 📌 Paso 3: Muestra Didáctica para Geometría en ℝ³

Para visualizar la geometría, necesitamos vectores en $\mathbb{R}^3$. Extraemos una muestra de **solo** $n=3$ individuos de nuestra simulación.

Usamos un bucle para garantizar que los 3 vectores formen una geometría no degenerada (ángulos adecuados, residuos proporcionales).

**Importante:** Restamos la media de las variables para trabajar con vectores que nacen del origen $(0,0,0)$. Esto elimina el intercepto y nos deja con la variación pura.

In [6]:
# Paso 3: Muestra Didáctica para Geometría en ℝ³
np.random.seed(123)

encontrado = False
intentos = 0

while not encontrado and intentos < 2000:
    intentos += 1
    idx_temp = np.random.choice(N, 3, replace=False)
    df_temp = df_mincer.iloc[idx_temp]

    # Desmeanar
    y_t = df_temp['ln_salario'].values - df_temp['ln_salario'].mean()
    x1_t = df_temp['educacion'].values - df_temp['educacion'].mean()
    x2_t = df_temp['experiencia'].values - df_temp['experiencia'].mean()

    # Criterios de calidad geométrica
    if np.std(y_t) < 0.2 or np.std(x1_t) < 0.2 or np.std(x2_t) < 0.2:
        continue

    cos_theta = np.dot(x1_t, x2_t) / (np.linalg.norm(x1_t) * np.linalg.norm(x2_t))
    if abs(cos_theta) > 0.65:  # Evitar multicolinealidad extrema
        continue

    # Proyecciones simples
    beta_1_geom_temp = np.dot(y_t, x1_t) / np.dot(x1_t, x1_t)
    yhat_1_temp = beta_1_geom_temp * x1_t
    e1_temp = y_t - yhat_1_temp

    beta_2_geom_temp = np.dot(y_t, x2_t) / np.dot(x2_t, x2_t)
    yhat_2_temp = beta_2_geom_temp * x2_t
    e2_temp = y_t - yhat_2_temp

    # Ratios de residuos
    norm_y = np.linalg.norm(y_t)
    ratio1 = np.linalg.norm(e1_temp) / norm_y
    ratio2 = np.linalg.norm(e2_temp) / norm_y

    if ratio1 < 0.2 or ratio1 > 0.9 or ratio2 < 0.2 or ratio2 > 0.9:
        continue

    # Guardar muestra válida
    y_c = y_t
    x1_c = x1_t
    x2_c = x2_t
    yhat_1 = yhat_1_temp
    e1 = e1_temp
    yhat_2 = yhat_2_temp
    e2 = e2_temp

    encontrado = True

print("=" * 50)
print("VERIFICACIÓN GEOMÉTRICA")
print("=" * 50)
print(f"Condición x₁ᵀe₁ = {np.dot(x1_c, e1):.10f} (≈ 0)")
print(f"Condición x₂ᵀe₂ = {np.dot(x2_c, e2):.10f} (≈ 0)")
print(f"\nMuestra encontrada en {intentos} intentos")

VERIFICACIÓN GEOMÉTRICA
Condición x₁ᵀe₁ = -0.0000000000 (≈ 0)
Condición x₂ᵀe₂ = -0.0000000000 (≈ 0)

Muestra encontrada en 3 intentos


# 📌 Paso 4: Preparación para Visualización 3D

Escalamos los vectores para que quepan bien en el gráfico 3D y calculamos las rectas que representan los subespacios generados por cada vector.

In [12]:
# Paso 4: Preparación para Visualización 3D (ESCALA AJUSTADA)
zoom = 8
max_v = max(np.abs(np.concatenate([x1_c, x2_c, y_c])))

X1s = (x1_c / max_v) * zoom
X2s = (x2_c / max_v) * zoom
Ys = (y_c / max_v) * zoom
P1s = (yhat_1 / max_v) * zoom
P2s = (yhat_2 / max_v) * zoom

# Rectas que representan span{x₁} y span{x₂}
lim = 3.5  # Aumentado de 2.5 a 3.5
recta_x1 = np.array([
    [-lim * X1s[0], lim * X1s[0]],
    [-lim * X1s[1], lim * X1s[1]],
    [-lim * X1s[2], lim * X1s[2]]
])

recta_x2 = np.array([
    [-lim * X2s[0], lim * X2s[0]],
    [-lim * X2s[1], lim * X2s[1]],
    [-lim * X2s[2], lim * X2s[2]]
])

print(f"Zoom: {zoom}")
print(f"Límite de rectas: {lim}")
print(f"Max valor escalado: {max_v}")

Zoom: 8
Límite de rectas: 3.5
Max valor escalado: 6.367938438792638


# 📌 Paso 5: Función Auxiliar para Flechas 3D

In [13]:
# Paso 5: Función Auxiliar para Flechas 3D (CORREGIDA)
def add_arrow3d(fig, from_pt, to_pt, color, name, lwd=4, cone_size=0.08, dash='solid'):
    """Agrega una flecha 3D al gráfico con tamaño proporcional"""
    dv = np.array(to_pt) - np.array(from_pt)
    dn = dv / np.linalg.norm(dv)  # Vector unitario de dirección

    # Línea de la flecha (más delgada)
    fig.add_trace(go.Scatter3d(
        x=[from_pt[0], to_pt[0] - dn[0]*0.05],
        y=[from_pt[1], to_pt[1] - dn[1]*0.05],
        z=[from_pt[2], to_pt[2] - dn[2]*0.05],
        mode='lines',
        line=dict(color=color, width=lwd, dash=dash),
        name=name,
        hoverinfo='text',
        text=name
    ))

    # Punta de la flecha (cono más pequeño)
    fig.add_trace(go.Cone(
        x=[to_pt[0]],
        y=[to_pt[1]],
        z=[to_pt[2]],
        u=[dn[0]],
        v=[dn[1]],
        w=[dn[2]],
        colorscale=[[0, color], [1, color]],
        showscale=False,
        sizemode='absolute',
        sizeref=cone_size,  # Tamaño reducido
        anchor='tip',
        hoverinfo='skip'
    ))

    return fig

print("Función add_arrow3d corregida con flechas proporcionales")

Función add_arrow3d corregida con flechas proporcionales


# 📌 Paso 6: Gráfico 1 - Doble Proyección 1D en ℝ³

Este gráfico muestra las proyecciones ortogonales simples de $\mathbf{y}$ sobre cada vector explicativo por separado.

Observa cómo cada residuo ($e_1$ y $e_2$) es perpendicular a su respectivo vector de proyección.

In [14]:
# Paso 6: Gráfico 1 - Doble Proyección 1D en ℝ³
origen = [0, 0, 0]

fig1 = go.Figure()

# Rectas de los subespacios
fig1.add_trace(go.Scatter3d(
    x=recta_x1[0], y=recta_x1[1], z=recta_x1[2],
    mode='lines',
    line=dict(color='#1e8449', width=2),
    name='span{x₁}',
    hoverinfo='skip'
))

fig1.add_trace(go.Scatter3d(
    x=recta_x2[0], y=recta_x2[1], z=recta_x2[2],
    mode='lines',
    line=dict(color='#6c3483', width=2),
    name='span{x₂}',
    hoverinfo='skip'
))

# Vectores originales
fig1 = add_arrow3d(fig1, origen, X1s, '#2ecc71', 'x₁: Educación')
fig1 = add_arrow3d(fig1, origen, X2s, '#9b59b6', 'x₂: Experiencia')
fig1 = add_arrow3d(fig1, origen, Ys, '#e74c3c', 'y: Log-Salario')

# Proyecciones
fig1 = add_arrow3d(fig1, origen, P1s, '#f39c12', 'Proy(y) en x₁')
fig1 = add_arrow3d(fig1, origen, P2s, '#00bcd4', 'Proy(y) en x₂')

# Residuos (líneas punteadas)
fig1.add_trace(go.Scatter3d(
    x=[P1s[0], Ys[0]], y=[P1s[1], Ys[1]], z=[P1s[2], Ys[2]],
    mode='lines',
    line=dict(color='#f39c12', width=3, dash='dash'),
    name='e₁'
))

fig1.add_trace(go.Scatter3d(
    x=[P2s[0], Ys[0]], y=[P2s[1], Ys[1]], z=[P2s[2], Ys[2]],
    mode='lines',
    line=dict(color='#00bcd4', width=3, dash='dash'),
    name='e₂'
))

fig1.update_layout(
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    scene=dict(
        aspectmode='cube',
        xaxis=dict(title='Ind 1', color='white'),
        yaxis=dict(title='Ind 2', color='white'),
        zaxis=dict(title='Ind 3', color='white')
    ),
    title=dict(text='Doble Proyección 1D en ℝ³', font=dict(color='white')),
    font=dict(color='white')
)

fig1.show()

# 📌 Paso 6: Gráfico 1 - Doble Proyección 1D en ℝ³

Este gráfico muestra las proyecciones ortogonales simples de $\mathbf{y}$ sobre cada vector explicativo por separado.

Observa cómo cada residuo ($e_1$ y $e_2$) es perpendicular a su respectivo vector de proyección.

In [15]:
# Paso 6: Gráfico 1 - Doble Proyección 1D en ℝ³
origen = [0, 0, 0]

fig1 = go.Figure()

# Rectas de los subespacios
fig1.add_trace(go.Scatter3d(
    x=recta_x1[0], y=recta_x1[1], z=recta_x1[2],
    mode='lines',
    line=dict(color='#1e8449', width=2),
    name='span{x₁}',
    hoverinfo='skip'
))

fig1.add_trace(go.Scatter3d(
    x=recta_x2[0], y=recta_x2[1], z=recta_x2[2],
    mode='lines',
    line=dict(color='#6c3483', width=2),
    name='span{x₂}',
    hoverinfo='skip'
))

# Vectores originales
fig1 = add_arrow3d(fig1, origen, X1s, '#2ecc71', 'x₁: Educación')
fig1 = add_arrow3d(fig1, origen, X2s, '#9b59b6', 'x₂: Experiencia')
fig1 = add_arrow3d(fig1, origen, Ys, '#e74c3c', 'y: Log-Salario')

# Proyecciones
fig1 = add_arrow3d(fig1, origen, P1s, '#f39c12', 'Proy(y) en x₁')
fig1 = add_arrow3d(fig1, origen, P2s, '#00bcd4', 'Proy(y) en x₂')

# Residuos (líneas punteadas)
fig1.add_trace(go.Scatter3d(
    x=[P1s[0], Ys[0]], y=[P1s[1], Ys[1]], z=[P1s[2], Ys[2]],
    mode='lines',
    line=dict(color='#f39c12', width=3, dash='dash'),
    name='e₁'
))

fig1.add_trace(go.Scatter3d(
    x=[P2s[0], Ys[0]], y=[P2s[1], Ys[1]], z=[P2s[2], Ys[2]],
    mode='lines',
    line=dict(color='#00bcd4', width=3, dash='dash'),
    name='e₂'
))

fig1.update_layout(
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    scene=dict(
        aspectmode='cube',
        xaxis=dict(title='Ind 1', color='white'),
        yaxis=dict(title='Ind 2', color='white'),
        zaxis=dict(title='Ind 3', color='white')
    ),
    title=dict(text='Doble Proyección 1D en ℝ³', font=dict(color='white')),
    font=dict(color='white')
)

fig1.show()

# 📌 Paso 7: Gráfico 2 - Teorema Frisch-Waugh-Lovell en ℝ³

Este gráfico ilustra el proceso de "purificación" del teorema FWL.

Los vectores naranjas y cyan muestran las variables una vez eliminada la influencia del control.

El vector verde grueso es la proyección final (el estimador $\hat{\beta}_1$).

In [21]:
# 📌 Paso 16: Gráfico 2 - Teorema FWL en ℝ³ (VERSIÓN LIMPIA)

# Función simplificada: solo líneas con puntos al final
def add_vector3d(fig, from_pt, to_pt, color, name, lwd=4, marker_size=6, dash='solid'):
    """Dibuja vectores 3D como líneas limpias con marcadores"""
    from_pt = np.array(from_pt)
    to_pt = np.array(to_pt)

    # Línea del vector
    fig.add_trace(go.Scatter3d(
        x=[from_pt[0], to_pt[0]],
        y=[from_pt[1], to_pt[1]],
        z=[from_pt[2], to_pt[2]],
        mode='lines+markers',
        line=dict(color=color, width=lwd, dash=dash),
        marker=dict(size=marker_size, color=color, symbol='circle'),
        name=name,
        hoverinfo='text',
        text=name,
        showlegend=True
    ))

    return fig

# --- 1. Cálculo matemático FWL ---
beta_y_x2  = np.dot(y_c, x2_c) / np.dot(x2_c, x2_c)
beta_x1_x2 = np.dot(x1_c, x2_c) / np.dot(x2_c, x2_c)

M1y  = y_c  - beta_y_x2  * x2_c
M1x1 = x1_c - beta_x1_x2 * x2_c

beta_fwl_1 = np.dot(M1y, M1x1) / np.dot(M1x1, M1x1)
P_FWL_1    = beta_fwl_1 * M1x1

# --- 2. Normalización Visual ---
vectors = [y_c, x1_c, x2_c, M1y, M1x1, P_FWL_1]
max_len = max(np.linalg.norm(v) for v in vectors)
k = 2.5 / max_len

X1s_viz = x1_c * k
X2s_viz = x2_c * k
Ys_viz  = y_c  * k
M1y_viz = M1y  * k
M1x1_viz = M1x1 * k
P_FWL_viz = P_FWL_1 * k

# --- 3. Renderizado Limpio ---
fig2 = go.Figure()

# Vectores originales (finos y atenuados)
fig2 = add_vector3d(fig2, [0,0,0], X1s_viz, '#7f8c8d', 'x₁ Original', lwd=2, marker_size=5)
fig2 = add_vector3d(fig2, [0,0,0], X2s_viz, '#95a5a6', 'x₂ Control', lwd=2, marker_size=5)
fig2 = add_vector3d(fig2, [0,0,0], Ys_viz, '#e74c3c', 'y Original', lwd=3, marker_size=6)

# Vectores Purificados FWL (más gruesos y destacados)
fig2 = add_vector3d(fig2, [0,0,0], M1y_viz, '#f39c12', 'M₁y: Salario Purificado', lwd=5, marker_size=8)
fig2 = add_vector3d(fig2, [0,0,0], M1x1_viz, '#00bcd4', 'M₁x₁: Educación Purificada', lwd=5, marker_size=8)

# Proyección Final (la más destacada)
fig2 = add_vector3d(fig2, [0,0,0], P_FWL_viz, '#2ecc71', 'β₁ FWL', lwd=7, marker_size=10)

fig2.update_layout(
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    title=dict(text='Teorema FWL: Purificación Geométrica', font=dict(color='white', size=16)),
    font=dict(color='white'),

    scene = dict(
        aspectmode='cube',
        xaxis=dict(range=[-3.5, 3.5], title='Ind 1', color='white', backgroundcolor="#111111", gridcolor="#333333"),
        yaxis=dict(range=[-3.5, 3.5], title='Ind 2', color='white', backgroundcolor="#111111", gridcolor="#333333"),
        zaxis=dict(range=[-3.5, 3.5], title='Ind 3', color='white', backgroundcolor="#111111", gridcolor="#333333"),

        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.2),
            center=dict(x=0, y=0, z=0),
            up=dict(x=0, y=0, z=1)
        )
    )
)

fig2.show()

print(f"β₁ (FWL) = {beta_fwl_1:.6f}")
print(f"Factor de escala (k) = {k:.4f}")

β₁ (FWL) = -0.078211
Factor de escala (k) = 0.2870


# 📌 Paso 8: Gráfico 3 - Regresión Múltiple (Proyección sobre el Plano)

Finalmente, este gráfico muestra la regresión múltiple completa: la proyección ortogonal de $\mathbf{y}$ sobre el plano generado por $\mathbf{x}_1$ y $\mathbf{x}_2$.

El vector $\hat{\mathbf{y}}$ es la suma de las componentes $\beta_1\mathbf{x}_1$ y $\beta_2\mathbf{x}_2$.

In [24]:
# Paso 8: Gráfico 3 - Regresión Múltiple (Proyección sobre el Plano)

# Calcular MCO usando álgebra lineal pura
X_mat = np.column_stack([x1_c, x2_c])
Y_vec = y_c
betas_mco_3d = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ Y_vec
b1_mco = betas_mco_3d[0]
b2_mco = betas_mco_3d[1]

yhat_mco = b1_mco * x1_c + b2_mco * x2_c
comp_x1 = b1_mco * x1_c
comp_x2 = b2_mco * x2_c

# Escalar vectores (tu código original)
Yhat_s = (yhat_mco / max_v) * zoom
C1_s = (comp_x1 / max_v) * zoom
C2_s = (comp_x2 / max_v) * zoom

fig3 = go.Figure()

# Vectores base
fig3 = add_arrow3d(fig3, origen, X1s, '#27ae60', 'x₁', lwd=4)
fig3 = add_arrow3d(fig3, origen, X2s, '#8e44ad', 'x₂', lwd=4)
fig3 = add_arrow3d(fig3, origen, Ys, '#e74c3c', 'y', lwd=6)

# Componentes de la proyección
fig3 = add_arrow3d(fig3, origen, C1_s, '#2ecc71', 'β₁x₁', lwd=7)
fig3 = add_arrow3d(fig3, origen, C2_s, '#9b59b6', 'β₂x₂', lwd=7)

# Proyección completa
fig3 = add_arrow3d(fig3, origen, Yhat_s, '#f1c40f', 'ŷ (Proyección)', lwd=8)

fig3.update_layout(
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    scene=dict(
        aspectmode='cube',
        # === AJUSTES PARA CENTRAR ===
        xaxis=dict(range=[-12, 12], title='Ind 1', color='white', backgroundcolor="#111111", gridcolor="#333333"),
        yaxis=dict(range=[-12, 12], title='Ind 2', color='white', backgroundcolor="#111111", gridcolor="#333333"),
        zaxis=dict(range=[-12, 12], title='Ind 3', color='white', backgroundcolor="#111111", gridcolor="#333333"),

        # Cámara fija centrada en el origen
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.2),
            center=dict(x=0, y=0, z=0),
            up=dict(x=0, y=0, z=1)
        )
    ),
    title=dict(text='Regresión Múltiple - Proyección sobre el Plano', font=dict(color='white')),
    font=dict(color='white')
)

fig3.show()

print("=" * 50)
print("PARÁMETROS MCO TRADICIONAL (en ℝ³)")
print("=" * 50)
print(f"β₁ = {b1_mco:.6f}")
print(f"β₂ = {b2_mco:.6f}")

PARÁMETROS MCO TRADICIONAL (en ℝ³)
β₁ = -0.078211
β₂ = 0.032677


#  Paso 9: Conclusión de la Simulación

Hemos demostrado tres cosas fundamentales que respaldan el paper teórico:

**1. Equivalencia Numérica (Paso 2):** Los coeficientes obtenidos por MCO múltiple son numéricamente idénticos a los obtenidos mediante el procedimiento de dos pasos de Frisch-Waugh-Lovell.

**2. Equivalencia Geométrica - Proyecciones Simples (Paso 6):** Las proyecciones ortogonales individuales muestran cómo cada variable explicativa captura parte de la variación de $\mathbf{y}$, pero los residuos ($e_1$, $e_2$) aún contienen información explicada por la otra variable.

**3. Equivalencia Geométrica - FWL (Paso 7):** El teorema FWL ilustra que "controlar por experiencia" es una operación geométrica concreta: proyectar ortogonalmente los vectores de interés sobre el complemento ortogonal del espacio generado por la variable de control.

**4. Equivalencia Geométrica - Regresión Múltiple (Paso 8):** La regresión múltiple completa proyecta $\mathbf{y}$ sobre el plano generado por $\mathbf{x}_1$ y $\mathbf{x}_2$. El estimador $\hat{\mathbf{y}}$ es la suma de las componentes $\beta_1\mathbf{x}_1 + \beta_2\mathbf{x}_2$, y el residuo final es perpendicular a todo el plano.

En síntesis, el estimador $\hat{\beta}_1$ es simplemente el factor de escala que alinea el vector de educación purificado ($M_1x_1$) con la componente del salario purificado ($M_1y$) que este logra explicar. La geometría en $\mathbb{R}^3$ hace visible lo que las fórmulas algebraicas solo sugieren: **minimizar es proyectar, y controlar es ortogonalizar**.